# Phase 4: Line and Word Segmentation Only

Use this notebook when the handwritten prescription regions are already cropped/segmented. It skips full-page preprocessing and region detection.

Input: `region_manifest.csv` or a folder of region crop images.

Output: line crops, word crops, numbered context images, paper/PPT-ready overview images, and review CSV files.

## Step 1: Mount Drive, Pull Latest Code, and Enter Repo

Run this first in Colab. It pulls the latest GitHub code, including the improved segmentation scripts.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = 'https://github.com/nbl-ahmd/project.git'
DRIVE_BASE = Path('/content/drive/MyDrive/phase4_project')
REPO_DIR = DRIVE_BASE / 'repo'
IN_COLAB = 'google.colab' in sys.modules or 'COLAB_GPU' in os.environ

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_BASE.mkdir(parents=True, exist_ok=True)
    if not (REPO_DIR / 'pipeline').exists():
        subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=False)
else:
    REPO_DIR = Path.cwd()

os.chdir(REPO_DIR)
print('Repository:', Path.cwd())
print('Has pipeline:', (Path.cwd() / 'pipeline').exists())

## Step 2: Install Dependencies

In [ ]:
!python3 -m pip install -q -r pipeline/requirements.txt

## Step 3: Set Region Input and Output Paths

Recommended: use an existing `region_manifest.csv`.

If you only have a folder of region crop images, set `REGION_MANIFEST = None` and set `REGION_CROPS_DIR` to that folder.

In [ ]:
from pathlib import Path

# Option A: use existing region manifest. Change this if your Drive path is different.
REGION_MANIFEST = Path('data/processed/raw_streamlit/region_manifest.csv')

# Option B: use only a region crop folder. Use this only if REGION_MANIFEST does not exist.
REGION_CROPS_DIR = Path('data/processed/raw_streamlit/regions')

OUTPUT_DIR = Path('data/line_word_segmentation_only')
LINE_DIR = OUTPUT_DIR / 'line_crops'
WORD_DIR = OUTPUT_DIR / 'word_crops'
OVERVIEW_DIR = OUTPUT_DIR / 'segmentation_overview'
LINE_MANIFEST = OUTPUT_DIR / 'line_manifest.csv'
WORD_MANIFEST = OUTPUT_DIR / 'word_manifest.csv'
LINE_REVIEW = OUTPUT_DIR / 'line_segmentation_review.csv'
WORD_REVIEW = OUTPUT_DIR / 'word_segmentation_review.csv'

for p in [OUTPUT_DIR, LINE_DIR, WORD_DIR, OVERVIEW_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print('Region manifest:', REGION_MANIFEST, 'exists=', REGION_MANIFEST.exists())
print('Region crops folder:', REGION_CROPS_DIR, 'exists=', REGION_CROPS_DIR.exists())
print('Output dir:', OUTPUT_DIR)

## Step 4: Create Region Manifest If You Only Have Region Images

Skip this if your `REGION_MANIFEST` already exists.

In [ ]:
import cv2
import pandas as pd
from pathlib import Path

if not REGION_MANIFEST.exists():
    rows = []
    image_paths = []
    for ext in ['*.png', '*.jpg', '*.jpeg', '*.webp', '*.bmp']:
        image_paths.extend(sorted(REGION_CROPS_DIR.glob(ext)))
    if not image_paths:
        raise FileNotFoundError(f'No region crop images found in {REGION_CROPS_DIR}')
    generated_manifest = OUTPUT_DIR / 'generated_region_manifest.csv'
    for i, path in enumerate(image_paths):
        img = cv2.imread(str(path))
        if img is None:
            continue
        h, w = img.shape[:2]
        region_id = path.stem if path.stem else f'region_{i:04d}'
        page_id = region_id.rsplit('_r', 1)[0] if '_r' in region_id else region_id
        rows.append({
            'page_id': page_id,
            'region_id': region_id,
            'region_image_path': str(path),
            'page_image_path': '',
            'x1_page': 0,
            'y1_page': 0,
            'x2_page': w,
            'y2_page': h,
        })
    pd.DataFrame(rows).to_csv(generated_manifest, index=False)
    REGION_MANIFEST = generated_manifest
    print('Created generated manifest:', REGION_MANIFEST, 'rows=', len(rows))
else:
    print('Using existing manifest:', REGION_MANIFEST)

## Step 5: Preview Region Crops

In [ ]:
import pandas as pd
from IPython.display import display, Image as IPImage

region_df = pd.read_csv(REGION_MANIFEST)
print('Regions:', len(region_df))
display(region_df.head())

for path in region_df['region_image_path'].dropna().astype(str).head(5):
    print(path)
    display(IPImage(filename=path, width=650))

## Step 6: Run Line Segmentation

This creates line crops and green numbered line-box overview images.

In [ ]:
!python3 pipeline/scripts/segment_lines.py \
  --region-manifest "{REGION_MANIFEST}" \
  --output-dir "{LINE_DIR}" \
  --manifest-out "{LINE_MANIFEST}" \
  --review-out "{LINE_REVIEW}" \
  --overview-dir "{OVERVIEW_DIR}" \
  --min-line-height 14 \
  --merge-gap 8 \
  --line-padding 6

line_df = pd.read_csv(LINE_MANIFEST)
line_review_df = pd.read_csv(LINE_REVIEW)
print('Line crops:', len(line_df))
display(line_df.head())
display(line_review_df.head())

## Step 7: Preview Line Results

In [ ]:
from IPython.display import display, Image as IPImage

print('Line context previews')
for path in sorted((LINE_DIR / 'context').glob('*.png'))[:5]:
    print(path)
    display(IPImage(filename=str(path), width=800))

print('Paper/PPT line overview previews')
for path in sorted(OVERVIEW_DIR.glob('*line_overview.png'))[:5]:
    print(path)
    display(IPImage(filename=str(path), width=900))

## Step 8: Run Word Segmentation

This creates word crops and red numbered word-box overview images.

In [ ]:
!python3 pipeline/scripts/segment_words.py \
  --line-manifest "{LINE_MANIFEST}" \
  --output-dir "{WORD_DIR}" \
  --manifest-out "{WORD_MANIFEST}" \
  --review-out "{WORD_REVIEW}" \
  --overview-dir "{OVERVIEW_DIR}" \
  --min-word-width 16 \
  --min-word-height 8 \
  --merge-gap 8 \
  --word-padding 3

word_df = pd.read_csv(WORD_MANIFEST)
word_review_df = pd.read_csv(WORD_REVIEW)
print('Word crops:', len(word_df))
display(word_df.head())
display(word_review_df.head())

## Step 9: Preview Word Results

In [ ]:
from IPython.display import display, Image as IPImage

print('Word context previews')
for path in sorted((WORD_DIR / 'context').glob('*.png'))[:8]:
    print(path)
    display(IPImage(filename=str(path), width=800))

print('Paper/PPT word overview previews')
for path in sorted(OVERVIEW_DIR.glob('*word_overview.png'))[:8]:
    print(path)
    display(IPImage(filename=str(path), width=900))

## Step 10: Inspect Problem Cases

Start by checking rows where no words or no lines were detected. Only annotate boxes manually if many of these are wrong.

In [ ]:
print('Line issues')
display(line_review_df[line_review_df['quality_flag'].astype(str) != 'ok'].head(20))

print('Word issues')
display(word_review_df[word_review_df['quality_flag'].astype(str) != 'ok'].head(20))

print('Useful output folders')
print('Line crops:', LINE_DIR / 'lines')
print('Word crops:', WORD_DIR / 'words')
print('Line contexts:', LINE_DIR / 'context')
print('Word contexts:', WORD_DIR / 'context')
print('Overview images:', OVERVIEW_DIR)

## Optional: When Manual Boxes Are Needed

Do not draw manual word boxes first. Use automatic segmentation and inspect the overview images.

Manual boxes are needed only if many lines/words are badly split or merged.

Recommended order:
1. Correct region boxes if region crops are bad.
2. Correct line boxes only for failed line crops.
3. Label word text/medicine names for clean word crops.
4. Train a word detector later only if automatic word boxes fail on many prescriptions.

For the current thesis implementation, use the auto-generated `word_manifest.csv` and review the word crops for OCR training.